In [81]:
import os
from dotenv import load_dotenv
import requests



In [82]:
base = "F:\data_water"
os.makedirs(base, exist_ok = True)

paths = ["raw", "processed"]
for fol in paths:
    os.makedirs(os.path.join(base, fol), exist_ok = True)


In [83]:
study_area = "Crete, Greece"
print(f"The study area chosen is a small part of southern {study_area}")

footprint = "POLYGON((23.9618338158908 35.2134473608656 0,23.936357546018 34.8574688986169 0,24.6072544958783 34.8016863349881 0,24.6482705348112 35.1518854484458 0,23.9618338158908 35.2134473608656 0))"

The study area chosen is a small part of southern Crete, Greece


In [89]:
def get_token():

    load_dotenv("../.env")        # reads .env file

    username = os.getenv("COPERNICUS_USERNAME")
    password = os.getenv("COPERNICUS_PASSWORD")
    data = {"client_id": "cdse-public", "username": username, "password": password, "grant_type": "password"}
    url = 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token'           # url for access token

    response = requests.post(url, data = data)
    print(response.status_code)
    token = response.json()['access_token']

    return token

In [90]:
token = get_token()
print(token[:20])

200
eyJhbGciOiJSUzI1NiIs


In [91]:
# Complete filters for OData (Open Data Protocol) based search for scenes or tiles

base_url = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products?$filter="

## SENTINEL 2 FILTERS
tile_filter = "Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'tileId' and att/OData.CSC.StringAttribute/Value eq 'T34SGE')"   #SINCE WHOLE STUDY AREA WAS FOUND TO BE INSIDE A TILE ITS FILTERED OUT AND DOWNLOADED
collection_filter_s2 = "Collection/Name eq 'SENTINEL-2'"
product_type_s2 = "Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' and att/OData.CSC.StringAttribute/Value eq 'S2MSI2A')"
cloud_cover = "Attributes/OData.CSC.DoubleAttribute/any(att:att/Name eq 'cloudCover' and att/OData.CSC.DoubleAttribute/Value lt 5.00)"
date_filter = "ContentDate/Start gt 2024-01-01T00:00:00.000Z and ContentDate/Start lt 2025-12-31T00:00:00.000Z"
area_filter = f"OData.CSC.Intersects(area=geography'SRID=4326;{footprint}')"
limit = "&$top=300"

In [92]:
SEARCH_FILTER = base_url + collection_filter_s2 + " and " + product_type_s2 + " and contains(Name,'T34SGE') and " + cloud_cover + " and " + date_filter + limit
headers = {"Authorization": f"Bearer {token}"}
search_response = requests.get(SEARCH_FILTER, headers=headers)
print(search_response.status_code)
print(search_response.text)
results = search_response.json()
scenes = results["value"]
print(len(scenes))
print(scenes[0]["Name"])

200
{"@odata.context":"$metadata#Products","value":[{"@odata.mediaContentType":"application/octet-stream","Id":"68fd87dc-c54a-48cd-8c71-901b2a74ca81","Name":"S2A_MSIL2A_20240330T090551_N0510_R050_T34SGE_20240330T134053.SAFE","ContentType":"application/octet-stream","ContentLength":901278654,"OriginDate":"2024-03-30T14:39:25.000000Z","PublicationDate":"2024-03-30T14:54:09.336914Z","ModificationDate":"2026-05-23T17:03:33.168081Z","Online":true,"EvictionDate":"9999-12-31T23:59:59.999999Z","S3Path":"/eodata/Sentinel-2/MSI/L2A/2024/03/30/S2A_MSIL2A_20240330T090551_N0510_R050_T34SGE_20240330T134053.SAFE","Checksum":[{"Value":"ced28ed1b9e06e62f0b70749065fda47","Algorithm":"MD5","ChecksumDate":"2026-05-23T17:03:31.076278Z"},{"Value":"52b20cba192d24b0bc5582246c67cb8a71daaf7a731b4f01b62f65251b59b72a","Algorithm":"BLAKE3","ChecksumDate":"2026-05-23T17:03:32.686790Z"}],"ContentDate":{"Start":"2024-03-30T09:05:51.024000Z","End":"2024-03-30T09:05:51.024000Z"},"Footprint":"geography'SRID=4326;POLYGON

In [36]:
t34sgd_scenes = [s for s in scenes if s["Name"][38:44] == "T34SGD"]
print(f"Total T34SGD scenes: {len(t34sgd_scenes)}")
# Print details of first T34SGD scene
scene = t34sgd_scenes[0]
print(scene.keys())  # first see what information is available

Total T34SGD scenes: 83
dict_keys(['@odata.mediaContentType', 'Id', 'Name', 'ContentType', 'ContentLength', 'OriginDate', 'PublicationDate', 'ModificationDate', 'Online', 'EvictionDate', 'S3Path', 'Checksum', 'ContentDate', 'Footprint', 'GeoFootprint'])


In [37]:
scene = t34sgd_scenes[0]
print(scene['Footprint'])
print(scene['Name'])
print(scene['ContentDate'])

geography'SRID=4326;POLYGON ((23.99879537752429 34.215009556583006, 24.361730334073396 34.206816537365285, 24.40199481397719 35.19525082926432, 24.267387824408175 35.19836573310012, 24.25877578111238 35.167647060152845, 24.218004917088955 35.01997120605731, 24.17736411020412 34.8723051103208, 24.1369673481414 34.72463259624277, 24.09667715089919 34.57701499813322, 24.056662317105275 34.42940578773496, 24.0167776302356 34.28183748988778, 23.99879537752429 34.215009556583006))'
S2A_MSIL2A_20230221T090001_N0510_R007_T34SGD_20240814T155244.SAFE
{'Start': '2023-02-21T09:00:01.024000Z', 'End': '2023-02-21T09:00:01.024000Z'}


In [93]:
for i, tiles in enumerate(scenes):                                  # the tokens are refreshed after 5 downloads so it wont get canceled in between
  if i % 10 == 0:
    token = get_token()
  headers = {"Authorization": f"Bearer {token}"}
  ids = tiles["Id"]
  url = f"https://download.dataspace.copernicus.eu/odata/v1/Products({ids})/$value"
  filename = tiles["Name"] + ".zip"
  filepath = os.path.join(base,  paths[0], filename)
  if os.path.exists(filepath):
    print(f"{tiles['Name']}, Already Downloaded")
    continue
  down_req = requests.get(url, headers = headers, allow_redirects=True, stream = True)
  if down_req.status_code == 200:
    with open(filepath, "wb") as f:
      for chunk in down_req.iter_content(chunk_size=8192):
        if chunk:
            f.write(chunk)
    print(f"{tiles['Name']}, Downloaded")
  else:
      print(f"Failed: {tiles['Name']}, status: {down_req.status_code}")

print("All scenes from SENTINEL-2 downloaded")

200
S2A_MSIL2A_20240330T090551_N0510_R050_T34SGE_20240330T134053.SAFE, Downloaded
S2A_MSIL2A_20240330T090551_N0510_R050_T34SGE_20240330T121454.SAFE, Downloaded
S2A_MSIL2A_20240127T090231_N0510_R007_T34SGE_20240127T121650.SAFE, Downloaded
S2A_MSIL2A_20240206T090141_N0510_R007_T34SGE_20240206T121453.SAFE, Downloaded
S2A_MSIL2A_20240117T090311_N0510_R007_T34SGE_20240117T121052.SAFE, Downloaded
S2B_MSIL2A_20240322T085629_N0510_R007_T34SGE_20240322T114528.SAFE, Downloaded
S2A_MSIL2A_20240310T090801_N0510_R050_T34SGE_20240310T122650.SAFE, Downloaded
S2B_MSIL2A_20240224T090859_N0510_R050_T34SGE_20240224T104230.SAFE, Downloaded
S2A_MSIL2A_20240327T085601_N0510_R007_T34SGE_20240327T122850.SAFE, Downloaded
S2B_MSIL2A_20240630T085559_N0510_R007_T34SGE_20240630T105345.SAFE, Downloaded
200
S2A_MSIL2A_20240608T090601_N0510_R050_T34SGE_20240608T122944.SAFE, Downloaded
S2B_MSIL2A_20240404T090549_N0510_R050_T34SGE_20240404T115134.SAFE, Downloaded
S2A_MSIL2A_20240506T085601_N0510_R007_T34SGE_20240506T13